<a href="https://colab.research.google.com/github/Kaushikraviiyer/EDA-Coursework/blob/main/Lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import os
import glob
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler

csv_file_path = "/content/fraudTest.csv"

if os.path.exists(csv_file_path):
    df = pd.read_csv(csv_file_path, low_memory=False)
else:
    np.random.seed(42)
    n_records = 1000
    all_amounts = np.concatenate([
        np.clip(np.random.normal(75, 50, int(n_records * 0.98)), 2.00, 500),
        np.random.uniform(1200, 9500, int(n_records * 0.02))
    ])
    np.random.shuffle(all_amounts)
    df = pd.DataFrame({"TransactionID": range(100001, 100001 + n_records), "amt": np.round(all_amounts, 2)})

target_col = "amt" if "amt" in df.columns else (df.columns[0] if "amt" not in df.columns else df.select_dtypes(include=[np.number]).columns[0])

print("=======================================================================")
print("                   DATASET PROFILE & INITIAL STATUS                    ")
print("=======================================================================")
print(f"Loaded File Path : {csv_file_path}")
print(f"Total Database Rows: {df.shape[0]}")
print(f"Target Feature     : {target_col}")
print("\nFirst 3 Target Rows Preview:")
print(df[[target_col]].head(3))
print("=======================================================================\n")

print("=======================================================================")
print("             EXP.5.1 - DISCRETIZATION AND BINNING                      ")
print("=======================================================================")
max_val = df[target_col].max() + 5000
risk_bins = [0, 50, 200, 1000, max_val]
risk_labels = ["Low Risk Tier", "Medium Risk Tier", "High Risk Tier", "Critical Risk/Flagged Fraud"]

df["FraudRiskCategory"] = pd.cut(df[target_col], bins=risk_bins, labels=risk_labels)
df["SpendQuartile"] = pd.qcut(df[target_col], q=4, labels=["Q1_Low", "Q2_Mid", "Q3_High", "Q4_Top"])

print("[Fixed-Width Binning Metrics via pd.cut]")
print(df["FraudRiskCategory"].value_counts())
print("\n[Quantile-Based Binning Metrics via pd.qcut]")
print(df["SpendQuartile"].value_counts())
print("=======================================================================\n")

print("=======================================================================")
print("          EXP.5.2 - FEATURE SCALING AND NORMALIZATION                  ")
print("=======================================================================")
scaler_standard = StandardScaler()
df["amt_ZScore_Scaled"] = scaler_standard.fit_transform(df[[target_col]])

scaler_minmax = MinMaxScaler()
df["amt_MinMax_Normalized"] = scaler_minmax.fit_transform(df[[target_col]])

print("Feature Distribution Transformation Outcomes:")
print(df[[target_col, "amt_ZScore_Scaled", "amt_MinMax_Normalized"]].head(5).to_string(index=False))
print("=======================================================================\n")

print("=======================================================================")
print("                    EXP.5.3 - OUTLIER DETECTION                        ")
print("=======================================================================")
outlier_threshold = 1200
outlier_mask = np.abs(df[target_col]) > outlier_threshold
suspected_fraud_df = df[outlier_mask]

print(f"Anomaly Cutoff Criterion            : Values > ${outlier_threshold}")
print(f"Extracted Anomaly Instances Volume : {len(suspected_fraud_df)}")

if not suspected_fraud_df.empty:
    display_cols = ["trans_date_trans_time", "cc_num", "merchant", "category", target_col, "FraudRiskCategory"]
    valid_display = [c for c in display_cols if c in df.columns]
    if len(valid_display) <= 1:
        valid_display = [target_col, "FraudRiskCategory", "amt_ZScore_Scaled"]

    print("\nIsolated Anomaly Subset Snapshot:")
    print(suspected_fraud_df[valid_display].head(10).to_string(index=False))
else:
    print("\nNo records breached the defined outlier criteria threshold.")
print("=======================================================================")


                   DATASET PROFILE & INITIAL STATUS                    
Loaded File Path : /content/fraudTest.csv
Total Database Rows: 555719
Target Feature     : amt

First 3 Target Rows Preview:
     amt
0   2.86
1  29.84
2  41.28

             EXP.5.1 - DISCRETIZATION AND BINNING                      
[Fixed-Width Binning Metrics via pd.cut]
FraudRiskCategory
Low Risk Tier                  288930
Medium Risk Tier               241440
High Risk Tier                  23766
Critical Risk/Flagged Fraud      1583
Name: count, dtype: int64

[Quantile-Based Binning Metrics via pd.qcut]
SpendQuartile
Q1_Low     139027
Q3_High    138952
Q4_Top     138904
Q2_Mid     138836
Name: count, dtype: int64

          EXP.5.2 - FEATURE SCALING AND NORMALIZATION                  
Feature Distribution Transformation Outcomes:
  amt  amt_ZScore_Scaled  amt_MinMax_Normalized
 2.86          -0.424463               0.000082
29.84          -0.252337               0.001267
41.28          -0.179353            